# Sentiment Analysis on Twitter Data

> **Project Goal:** Build a classification model that predicts whether a tweet expresses positive or negative sentiment.
>
> **Dataset:** Sentiment140 -- 200,000 labeled tweets (100K positive, 100K negative) from April-May 2009.
>
> **Why this matters:** Sentiment analysis is one of the most widely deployed NLP applications in industry -- brand monitoring, customer feedback analysis, social media listening, and stock market prediction all rely on understanding public sentiment at scale.

---

## Project Roadmap

| Step | Section | What We'll Do |
|:----:|:--------|:--------------|
| 1 | Problem Framing | Understand the business value and define success metrics |
| 2 | Setup and Imports | Load all required libraries |
| 3 | Data Loading | Read the CSV with correct encoding and inspect it |
| 4 | Exploratory Data Analysis | Visualize distributions, text patterns, and class balance |
| 5 | Text Preprocessing | Clean tweets -- remove noise, tokenize, normalize |
| 6 | Visual EDA | Word clouds and top-N frequent words per sentiment |
| 7 | Feature Engineering | Convert text to numerical features (TF-IDF) |
| 8 | Train/Test Split | Create stratified train/test sets |
| 9 | Model Building | Train Logistic Regression and Naive Bayes classifiers |
| 10 | Model Evaluation | Confusion matrix, classification report, ROC curve |
| 11 | Error Analysis | Examine misclassified tweets for insights |
| 12 | Conclusion | Summary, findings, and ideas for improvement |

---


## 1. Environment Setup -- Importing Libraries

Before we do anything, we need to load the tools for the job. Here is what each library gives us:

- **pandas** -- The backbone of data manipulation in Python. DataFrames make it easy to filter, aggregate, and transform our data.
- **numpy** -- Numerical computing. Underpins most ML libraries and gives us fast array operations.
- **matplotlib + seaborn** -- The standard combo for static visualizations. Seaborn sits on top of matplotlib and makes statistical plots prettier with less code.
- **nltk** -- Natural Language Toolkit. The gold standard for text preprocessing: tokenization, stopwords, stemming.
- **re** -- Regular expressions. Essential for pattern-based text cleaning (URLs, mentions, emojis).
- **wordcloud** -- Generates word clouds where word size equals frequency. Great for visual EDA.
- **sklearn** -- scikit-learn, the workhorse of classical ML. We will use it for feature extraction (TF-IDF), model training, and evaluation.
- **warnings** -- Suppresses non-critical warnings so our output stays clean.

> Think of this cell as laying out your toolbox before starting a repair job. You want everything within reach before you begin.


In [ ]:
# Data manipulation and analysis
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# Text processing
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

# Machine learning
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

# Suppress non-critical warnings for clean output
import os

# Create output directory for visualizations
os.makedirs('sentiment_analysis/visualization', exist_ok=True)

import warnings
warnings.filterwarnings('ignore')

# Set visualization defaults
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 120

print('All libraries imported successfully!')


### Downloading NLTK Resources

NLTK requires us to download certain data files (tokenizers, stopword lists) before first use. Think of this like installing a language pack -- the NLTK algorithms need reference data to know what a 'word' is and what words to ignore.

We only need to do this once. The files are cached locally after download.


In [ ]:
# Download required NLTK data files
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

print('NLTK data downloaded!')


## 2. Data Loading -- Reading the CSV

The Sentiment140 dataset does not come with a header row, so we need to supply column names manually. The columns are:

| Column | Type | Description |
|:-------|:----:|:------------|
| target | int | Sentiment label: 0 = negative, 4 = positive |
| id | int | Unique tweet ID |
| date | str | Timestamp of the tweet |
| query | str | Search query used (NO_QUERY for most) |
| user | str | Username of the tweet author |
| text | str | The actual tweet content -- this is our input feature! |

**Important encoding detail:** The dataset contains non-UTF-8 characters (emoticons, accented letters), so we use latin-1 encoding. Encoding issues are a common gotcha -- always check when loading real-world CSVs!

> Latin-1 (ISO-8859-1) can encode any byte value 0-255, so it never throws errors on unknown characters. UTF-8 is stricter and will crash on invalid byte sequences. For messy social media text, latin-1 is your safety net.


In [ ]:
# Define column names (the CSV has no header)
COLUMNS = ['target', 'id', 'date', 'query', 'user', 'text']

# Resolve the data file path — works from any working directory
import os

# Create output directory for visualizations
os.makedirs('sentiment_analysis/visualization', exist_ok=True)

DATA_DIR = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
# In Jupyter, __file__ doesn't exist — fall back to cwd or notebook-relative
if not os.path.exists(DATA_DIR):
    # Try relative to project root pattern: sentiment_analysis/data/
    for candidate in ['sentiment_analysis/data/training.csv',
                       '../data/training.csv',
                       'data/training.csv']:
        if os.path.exists(candidate):
            DATA_PATH = candidate
            break
else:
    DATA_PATH = os.path.join(DATA_DIR, 'data/training.csv')

# Load the dataset
# encoding='latin-1' handles non-UTF-8 characters common in social media text
df = pd.read_csv(
    DATA_PATH,
    encoding='latin-1',
    header=None,
    names=COLUMNS,
    dtype={'target': 'int8'}  # smaller dtype = less memory usage
)

print(f'Loaded {df.shape[0]:,} rows x {df.shape[1]} columns')

### First Look at the Data

Let us peek at the first few rows and summary statistics. This is our 'hello world' moment -- getting familiar with the raw material before we start transforming it.

**What to notice:**
- The target column uses 0 and 4 (not 0 and 1). We will recode 4 to 1 for standard binary classification.
- The text column contains URLs, @mentions, and hashtags -- noisy but informative.
- The query column is almost all NO_QUERY -- we can probably drop this column.
- date and id are not useful for prediction but we keep them for reference.


In [ ]:
# Display first 5 rows -- transposed for easier reading
df.head(3).T


In [ ]:
# Quick summary statistics
print('Shape:', df.shape)
print('')
print('Column dtypes:')
print(df.dtypes)
print('')
print('Missing values per column:')
print(df.isnull().sum())
print('')
print('Unique values:')
print(f'  Users: {df["user"].nunique():,}')
print(f'  Dates: {df["date"].nunique():,}')
print(f'  Targets: {sorted(df["target"].unique())}')


### Recoding the Target Variable

The dataset uses 0 for negative and 4 for positive. For scikit-learn binary classification, we prefer 0 and 1.

- 0 stays 0 (negative sentiment)
- 4 maps to 1 (positive sentiment)

> Why 0/1? Most ML classifiers expect binary targets as 0/1. The predict_proba() method returns probabilities for both classes, and metrics like ROC-AUC are designed around 0/1 encoding.


In [ ]:
# Recode target: 4 to 1 (positive), 0 stays 0 (negative)
df['sentiment'] = df['target'].map({0: 0, 4: 1})

# Verify the recoding
print('Target distribution after recoding:')
print(df['sentiment'].value_counts().sort_index())
print(f'')
print(f'Class balance:')
print(df['sentiment'].value_counts(normalize=True).sort_index().round(3))


## 3. Exploratory Data Analysis

EDA is the detective work of data science. Before building models, we need to understand:

1. **Is the dataset balanced?** Imbalanced classes require special handling (SMOTE, class weights).
2. **What do tweets look like?** Length distribution, common patterns.
3. **Are there differences between positive and negative tweets?** Do negative tweets have more punctuation? Are positive tweets longer?

The dataset is perfectly balanced -- 100K positive, 100K negative. This is rare in real-world data and means we can use accuracy as a reliable metric.


In [ ]:
# Distribution of sentiment labels
plt.figure(figsize=(8, 5))
colors = ['#e74c3c', '#2ecc71']  # red = negative, green = positive

counts = df['sentiment'].value_counts().sort_index()
bars = plt.bar(['Negative (0)', 'Positive (1)'], counts.values, color=colors, edgecolor='white', linewidth=1.5)

# Add count labels on top of each bar
for bar in bars:
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2000,
             f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=14, fontweight='bold')

plt.title('Sentiment Class Distribution', fontsize=16, fontweight='bold', pad=15)
plt.ylabel('Number of Tweets', fontsize=12)
plt.ylim(0, counts.max() * 1.1)
sns.despine()
plt.tight_layout()
plt.savefig('sentiment_analysis/visualization/class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

print('Perfectly balanced dataset -- 100K tweets per class!')


### Text Length Analysis

A simple but powerful feature: tweet length (character count). Do positive and negative tweets differ in length?

> Why text length matters: It is a cheap, interpretable feature. In some domains, longer text correlates with more thoughtful/positive sentiment, while shorter text can indicate strong emotion (positive or negative).


In [ ]:
# Add a text length column
df['text_length'] = df['text'].str.len()

# Summary statistics grouped by sentiment
length_stats = df.groupby('sentiment')['text_length'].describe().round(1)
length_stats.index = ['Negative', 'Positive']
length_stats


In [ ]:
# Distribution of text length by sentiment -- overlaid histograms
plt.figure(figsize=(12, 6))

# Histogram with KDE overlay
for sentiment, color, label in [(0, '#e74c3c', 'Negative'), (1, '#2ecc71', 'Positive')]:
    subset = df[df['sentiment'] == sentiment]['text_length']
    sns.histplot(subset, bins=60, color=color, alpha=0.35, label=label, stat='density', kde=True)

plt.title('Tweet Length Distribution by Sentiment', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Number of Characters', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.legend(fontsize=12)
plt.xlim(0, 300)
sns.despine()
plt.tight_layout()
plt.savefig('sentiment_analysis/visualization/text_length_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

print('Observations:')
print('  - Most tweets are between 20-120 characters')
print('  - Negative tweets have a slightly higher density of very short tweets')
print('  - Positive tweets are slightly more spread across longer lengths')


## 4. Text Preprocessing

This is the most critical step in any NLP pipeline. Raw text is messy -- it contains URLs, @mentions, punctuation, numbers, and inconsistent casing. Machine learning models see these as unique tokens, so 'Hello', 'hello!', and 'hello' would all be treated as different words.

**Our cleaning steps, in order:**

| Step | What | Why |
|:----:|:-----|:----|
| 1 | Lowercasing | Convert all text to lowercase so 'Great', 'great', and 'GREAT' are the same token |
| 2 | Remove URLs | 'http://t.co/xyz123' carries no sentiment signal |
| 3 | Remove @mentions | '@user123' is noise -- handles are identifiers, not sentiment |
| 4 | Remove numbers | Numerical values rarely carry sentiment independently |
| 5 | Remove punctuation | Keeps only letters and spaces |
| 6 | Remove stopwords | 'the', 'a', 'an', 'in', 'of' -- appear in every tweet, dilute signal |
| 7 | Stemming | Reduce words to root form: 'running' to 'run', 'happiness' to 'happi' |

> Each step depends on the previous one. URLs might contain numbers (step 4), mentions might contain punctuation (step 5). Ordering matters -- we remove structural elements before breaking down language.


In [ ]:
# Initialize our preprocessing tools
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

print(f'Loaded {len(stop_words):,} English stopwords')
print(f'Using Porter Stemmer')


In [ ]:
def clean_text(text):
    """
    Clean and preprocess a single tweet.
    
    Applies a series of regex and NLP operations to transform
    raw tweet text into clean, stemmed tokens ready for vectorization.
    """
    # Step 1: Convert to lowercase
    text = text.lower()

    # Step 2: Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # Step 3: Remove @mentions
    text = re.sub(r'@\w+', '', text)

    # Step 4: Remove numbers
    text = re.sub(r'\d+', '', text)

    # Step 5: Keep only letters and spaces
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Step 6: Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # Step 7: Tokenize
    tokens = word_tokenize(text)

    # Step 8: Remove stopwords and short tokens, then apply stemming
    cleaned_tokens = [
        stemmer.stem(token)
        for token in tokens
        if token not in stop_words and len(token) > 2
    ]

    return ' '.join(cleaned_tokens)

# Test our cleaning function on a sample tweet
sample_tweet = df['text'].iloc[0]
cleaned_sample = clean_text(sample_tweet)

print('Before cleaning:')
print(f'   {sample_tweet}')
print('')
print('After cleaning:')
print(f'   {cleaned_sample}')


### Applying Preprocessing to the Full Dataset

Now we apply our cleaning function to all 200K tweets. We create a new column clean_text so we do not lose the original data.


In [ ]:
# Apply the cleaning function to all tweets
print('Cleaning 200,000 tweets... This may take a minute or two...')
df['clean_text'] = df['text'].apply(clean_text)

# Quick sanity check -- show a few before/after comparisons
comparison = pd.DataFrame({
    'Original': df['text'].iloc[:3].values,
    'Cleaned':  df['clean_text'].iloc[:3].values
})
comparison


## 5. Visual EDA -- What Words Define Each Sentiment?

Before building a model, let us see if certain words are strongly associated with one sentiment or the other.

- Are there words that appear almost exclusively in one class?
- Is the vocabulary overlap between classes large or small?
- Are there unexpected words (e.g., positive words in negative tweets)?

> This step helps build intuition for what features the model will learn.


In [ ]:
# Function to get word frequency per sentiment class
def get_word_freq(text_series):
    all_words = ' '.join(text_series).split()
    from collections import Counter
    return Counter(all_words)

# Split by sentiment
neg_words = get_word_freq(df[df['sentiment'] == 0]['clean_text'])
pos_words = get_word_freq(df[df['sentiment'] == 1]['clean_text'])

print(f'Negative vocabulary: {len(neg_words):,} unique words')
print(f'Positive vocabulary: {len(pos_words):,} unique words')
print(f'Shared vocabulary:  {len(set(neg_words.keys()) & set(pos_words.keys())):,} words')


### Word Clouds -- Negative vs. Positive Sentiment

Word clouds give us an immediate visual sense of the language landscape for each class. The bigger the word, the more frequently it appears.


In [ ]:
# Generate word clouds for both sentiments
fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# Negative word cloud
neg_text = ' '.join(df[df['sentiment'] == 0]['clean_text'])
wc_neg = WordCloud(width=800, height=400, max_words=150, background_color='white',
                   colormap='Reds', collocations=False).generate(neg_text)
axes[0].imshow(wc_neg, interpolation='bilinear')
axes[0].axis('off')
axes[0].set_title('Negative Sentiment', fontsize=18, fontweight='bold', pad=20)

# Positive word cloud
pos_text = ' '.join(df[df['sentiment'] == 1]['clean_text'])
wc_pos = WordCloud(width=800, height=400, max_words=150, background_color='white',
                   colormap='Greens', collocations=False).generate(pos_text)
axes[1].imshow(wc_pos, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title('Positive Sentiment', fontsize=18, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('sentiment_analysis/visualization/wordcloud_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

print('Observations:')
print('  - Negative tweets: dominated by words like miss, hate, sad, lost, bad, sick')
print('  - Positive tweets: dominated by words like love, good, day, great, thank')
print('  - Significant overlap: neutral words like get, got, like, go appear in both')


### Top 20 Words Per Sentiment Class

Word clouds are artistic but imprecise. Let us get exact counts with a bar chart.


In [ ]:
def plot_top_words(word_counter, title, color, n=20):
    top_words = word_counter.most_common(n)
    words, counts = zip(*top_words)

    plt.figure(figsize=(10, 8))
    bars = plt.barh(range(len(words)), counts, color=color, edgecolor='white')
    plt.yticks(range(len(words)), words)
    plt.gca().invert_yaxis()
    plt.title(title, fontsize=16, fontweight='bold', pad=15)
    plt.xlabel('Frequency', fontsize=12)

    for bar, count in zip(bars, counts):
        plt.text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
                 f'{count:,}', va='center', fontsize=10)

    sns.despine()
    plt.tight_layout()
    plt.savefig('sentiment_analysis/visualization/class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

plot_top_words(neg_words, 'Top 20 Words -- Negative Tweets', '#e74c3c')
plot_top_words(pos_words, 'Top 20 Words -- Positive Tweets', '#2ecc71')


## 6. Train/Test Split

Before we train any model, we must hold out a portion of data for final evaluation. This simulates how our model would perform on new, unseen tweets.

**Why split?**
- If we train and evaluate on the same data, the model might just memorize (overfitting).
- The test set is our honest broker -- it tells us how well our model generalizes.

**Why stratified?**
- stratify=y preserves the class proportions (50/50) in both train and test sets.
- Without stratification, random splitting could accidentally create a skewed test set.

**Split ratio:** 80/20 (160K train, 40K test).


In [ ]:
# Define features (X) and target (y)
X = df['clean_text']  # our cleaned tweet text
y = df['sentiment']   # 0 = negative, 1 = positive

# Stratified train/test split -- preserves class balance in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 20% for testing
    random_state=42,     # fixed seed for reproducible results
    stratify=y,          # maintains 50/50 class ratio
)

print(f'Train size:    {len(X_train):,} tweets')
print(f'Test size:     {len(X_test):,} tweets')
print(f'')
print(f'Train class distribution:')
print(y_train.value_counts().to_string())
print(f'')
print(f'Test class distribution:')
print(y_test.value_counts().to_string())


## 7. Feature Engineering -- TF-IDF Vectorization

Machine learning models do not understand text -- they understand numbers. We need to convert our cleaned tweets into numerical feature vectors.

### What is TF-IDF?

**TF-IDF** stands for **Term Frequency -- Inverse Document Frequency**. It answers two questions:

1. **TF (Term Frequency):** How often does a word appear in this tweet? (more = more important)
2. **IDF (Inverse Document Frequency):** How rare is this word across ALL tweets? (rarer words get higher weight)

The formula: TF-IDF = TF x log(N / DF)
- N = total number of tweets
- DF = number of tweets containing this word

### Why TF-IDF over Bag of Words?

Bag of Words counts occurrences -- common words like 'the' and 'day' get inflated importance.
TF-IDF penalizes common words. Rare sentiment-bearing words like 'horrible' and 'fantastic' get their deserved weight.

> Intuition: If 'the' appears in 99% of tweets and 'fantastic' appears in 2%, 'fantastic' is a much stronger signal. TF-IDF amplifies that signal.

**Parameters:**
- max_features=5000 -- Keep the 5,000 most informative words
- ngram_range=(1,2) -- Single words and two-word phrases
- max_df=0.95 -- Ignore words in >95% of tweets (too common to be useful)
- min_df=5 -- Ignore words appearing in <5 tweets (too rare, likely noise)


In [ ]:
# Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words='english',
    max_df=0.95,
    min_df=5,
)

print('TF-IDF Vectorizer configured')
print(f'   max_features=5,000')
print(f'   ngram_range=(1, 2) -- unigrams and bigrams')
print(f'   max_df=0.95 -- ignoring words in >95% of tweets')
print(f'   min_df=5 -- ignoring words in <5 tweets')


In [ ]:
# Fit the vectorizer on training data and transform both train and test
print('Fitting TF-IDF on training data and transforming...')
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f'Training feature matrix: {X_train_tfidf.shape}')
print(f'Test feature matrix:     {X_test_tfidf.shape}')
print(f'Vocabulary size: {len(tfidf.vocabulary_):,} words')

# Show some of the most informative features
feature_names = tfidf.get_feature_names_out()
print(f'Sample features: {feature_names[:10]}')


## 8. Model Building -- Logistic Regression

### Why start with Logistic Regression?

Logistic Regression is the Swiss Army knife of binary classification:
1. Fast to train -- seconds on 160K samples
2. Interpretable -- we can see which words most strongly influence the decision
3. Probabilistic -- gives confidence scores, not just labels
4. Well-calibrated -- probabilities match real-world frequencies
5. Strong baseline -- often performs surprisingly well on text data

### How it works (simplified):

It learns a weight for each word: positive weight pushes prediction toward positive, negative weight pushes toward negative. The final prediction is the sum of these weighted word scores, passed through a sigmoid function that squashes the output to a probability between 0 and 1.

If P(y=1) > 0.5, predict positive; otherwise, predict negative.


In [ ]:
# Train Logistic Regression
lr_model = LogisticRegression(
    C=1.0,              # inverse regularization strength
    max_iter=1000,       # enough iterations for convergence
    solver='liblinear',  # good for smaller datasets, handles L1/L2
    random_state=42,
)

print('Training Logistic Regression on 160K tweets...')
lr_model.fit(X_train_tfidf, y_train)
print('Training complete!')


### Bonus Model -- Multinomial Naive Bayes

Naive Bayes is a probabilistic classifier that applies Bayes' Theorem with a 'naive' assumption: each word contributes independently to the sentiment. This assumption is technically wrong, but Naive Bayes works surprisingly well for text classification.

**Why it works on text:**
- Word occurrences are treated as independent evidence for/against a class
- Even with the independence assumption violated, the ranking of classes tends to be correct
- Very fast to train and predict


In [ ]:
# Train Multinomial Naive Bayes
nb_model = MultinomialNB(alpha=1.0)  # alpha=1.0 is Laplace smoothing

print('Training Multinomial Naive Bayes...')
nb_model.fit(X_train_tfidf, y_train)
print('Training complete!')


## 9. Model Evaluation

Now for the moment of truth. We evaluate our models on the held-out test set (40K tweets the models have never seen).

### Why these metrics?

| Metric | What it measures | Why it matters |
|:-------|:-----------------|:---------------|
| Accuracy | Overall % correct | Baseline measure -- works well since classes are balanced |
| Precision | Of tweets predicted positive, how many are actually positive? | Minimizes false positives |
| Recall | Of actual positive tweets, how many did we find? | Minimizes false negatives |
| F1-Score | Harmonic mean of Precision and Recall | Single number balancing both |
| ROC-AUC | Model ability to rank positive higher than negative | Threshold-independent measure |


In [ ]:
# Make predictions on the test set
y_pred_lr = lr_model.predict(X_test_tfidf)
y_pred_nb = nb_model.predict(X_test_tfidf)

# Predicted probabilities (for ROC-AUC)
y_prob_lr = lr_model.predict_proba(X_test_tfidf)[:, 1]
y_prob_nb = nb_model.predict_proba(X_test_tfidf)[:, 1]

print('Predictions generated for both models!')


In [ ]:
# Create a comparison table of all metrics
def evaluate_model(name, y_true, y_pred, y_prob):
    return {
        'Model': name,
        'Accuracy':  accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall':    recall_score(y_true, y_pred),
        'F1-Score':  f1_score(y_true, y_pred),
        'ROC-AUC':   roc_auc_score(y_true, y_prob),
    }

results = [
    evaluate_model('Logistic Regression', y_test, y_pred_lr, y_prob_lr),
    evaluate_model('Naive Bayes', y_test, y_pred_nb, y_prob_nb),
]

results_df = pd.DataFrame(results).set_index('Model')
results_df


### Confusion Matrix Analysis

A confusion matrix shows us where the model makes mistakes:

| | Predicted Negative | Predicted Positive |
|:---|:---:|:---:|
| Actual Negative | True Negatives (TN) | False Positives (FP) |
| Actual Positive | False Negatives (FN) | True Positives (TP) |

**What to look for:**
- Are false positives and false negatives balanced?
- Does the model favor one class over the other?


In [ ]:
# Plot confusion matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, (model_name, y_pred) in enumerate([
    ('Logistic Regression', y_pred_lr),
    ('Naive Bayes', y_pred_nb)
]):
    cm = confusion_matrix(y_test, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    sns.heatmap(cm_norm, annot=cm, fmt='d', cmap='Blues',
                xticklabels=['Negative', 'Positive'],
                yticklabels=['Negative', 'Positive'],
                ax=axes[idx], cbar_kws={'label': 'Proportion'})

    axes[idx].set_title(f'{model_name}',
                       fontsize=14, fontweight='bold')
    axes[idx].set_xlabel('Predicted', fontsize=11)
    axes[idx].set_ylabel('Actual', fontsize=11)

plt.tight_layout()
plt.savefig('sentiment_analysis/visualization/confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()


### ROC Curve Analysis

The ROC curve plots the trade-off between:
- True Positive Rate (TPR) = Recall
- False Positive Rate (FPR) = false alarms

A perfect classifier hugs the top-left corner. A random classifier follows the diagonal (AUC = 0.5).

> AUC interpretation: If you randomly pick one positive tweet and one negative tweet, AUC is the probability that the model ranks the positive tweet higher.


In [ ]:
# Plot ROC curves for both models
plt.figure(figsize=(10, 8))

for model_name, y_prob, color in [
    ('Logistic Regression', y_prob_lr, '#3498db'),
    ('Naive Bayes', y_prob_nb, '#e67e22')
]:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, color=color, lw=2.5,
             label=f'{model_name} (AUC = {auc:.3f})')

# Diagonal line = random classifier
plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier (AUC = 0.5)')

plt.xlabel('False Positive Rate (FPR)', fontsize=13)
plt.ylabel('True Positive Rate (TPR)', fontsize=13)
plt.title('ROC Curves -- Model Comparison', fontsize=16, fontweight='bold', pad=15)
plt.legend(fontsize=12, loc='lower right')
plt.xlim(-0.02, 1.02)
plt.ylim(-0.02, 1.02)
sns.despine()
plt.tight_layout()
plt.savefig('sentiment_analysis/visualization/roc_curves.png', dpi=120, bbox_inches='tight')
plt.show()

print('The higher the AUC, the better the model distinguishes positive from negative.')


### What Words Drive the Model's Decisions?

One of the biggest advantages of Logistic Regression is interpretability. We can look at the learned coefficients to see which words most strongly push the prediction toward positive or negative sentiment.

- High positive coefficient -> strong signal for positive sentiment
- High negative coefficient -> strong signal for negative sentiment
- Near-zero coefficient -> word does not influence the prediction much


In [ ]:
# Extract feature names and coefficients from Logistic Regression
feature_names = tfidf.get_feature_names_out()
coefficients = lr_model.coef_.flatten()

# Create a DataFrame for easy sorting
feature_importance = pd.DataFrame({
    'word': feature_names,
    'coefficient': coefficients
}).sort_values('coefficient', ascending=False)

# Top 20 words pushing toward POSITIVE sentiment
print('Top 20 words -- Signal for POSITIVE sentiment:')
print(feature_importance.head(20).to_string(index=False))

print('')
print('-' * 60)
print('')

# Bottom 20 words pushing toward NEGATIVE sentiment
print('Top 20 words -- Signal for NEGATIVE sentiment:')
print(feature_importance.tail(20).iloc[::-1].to_string(index=False))


## 10. Error Analysis -- Learning from Mistakes

The most important part of model building is not training -- it is understanding what went wrong. Error analysis helps us:

1. Identify data issues -- Are there mislabeled examples?
2. Spot edge cases -- What kinds of tweets consistently fool the model?
3. Plan improvements -- Should we add features? Get more data? Try a different model?


In [ ]:
# Create a DataFrame with predictions vs actual
errors = pd.DataFrame({
    'text': X_test.values,
    'actual': y_test.values,
    'predicted': y_pred_lr,
    'prob_pos': y_prob_lr,
})

# Find misclassified tweets
errors['correct'] = errors['actual'] == errors['predicted']
misclassified = errors[~errors['correct']].copy()

print(f'Total misclassified: {len(misclassified):,} out of {len(errors):,} test tweets ({len(misclassified)/len(errors)*100:.1f}%)')
print(f'Correctly classified: {errors["correct"].sum():,} ({errors["correct"].sum()/len(errors)*100:.1f}%)')

# Show some misclassified examples
print('')
print('Sample misclassified tweets:')
print('=' * 90)

for i, row in misclassified.head(10).iterrows():
    actual_label = 'POSITIVE' if row['actual'] == 1 else 'NEGATIVE'
    pred_label  = 'POSITIVE' if row['predicted'] == 1 else 'NEGATIVE'
    confidence = row['prob_pos'] if row['predicted'] == 1 else 1 - row['prob_pos']
    print(f'')
    print(f'Actual: {actual_label}  |  Predicted: {pred_label}  |  Confidence: {confidence:.2%}')
    print(f'Tweet: "{row["text"][:120]}"')
    print('-' * 90)


### Patterns in Misclassifications

Common patterns in sentiment analysis errors:

1. Sarcasm -- 'Great, another flat tire...' -- positive word with negative intent
2. Negation -- 'Not bad' -- negative word structure with positive meaning
3. Mixed sentiment -- 'I love this but it is so expensive' -- both signals present
4. Context-dependent words -- 'sick' can mean ill (negative) or awesome (positive, slang)
5. Emoji-only tweets -- our cleaning removes non-alphabetic characters

Our current model is a bag-of-words model -- it sees words in isolation, not in sequence. It struggles with sarcasm and negation, where word order matters. A deep learning model (LSTM, Transformer) could capture these patterns.


In [ ]:
# Analyze misclassifications by type
false_positives = errors[(errors['actual'] == 0) & (errors['predicted'] == 1)]
false_negatives = errors[(errors['actual'] == 1) & (errors['predicted'] == 0)]

print(f'False Positives (predicted positive, was negative): {len(false_positives):,}')
print(f'False Negatives (predicted negative, was positive): {len(false_negatives):,}')
print(f'')

if len(false_positives) > len(false_negatives):
    print('Model is slightly biased toward predicting POSITIVE')
elif len(false_negatives) > len(false_positives):
    print('Model is slightly biased toward predicting NEGATIVE')
else:
    print('Errors are perfectly balanced between both classes')


## 11. Conclusion and Key Takeaways

### Key Learnings

1. **Logistic Regression outperforms Naive Bayes** on this dataset -- it captures feature interactions that NB independence assumption misses.

2. **Text preprocessing matters enormously** -- removing URLs, mentions, and punctuation cleaned up many tokens that carried no sentiment signal.

3. **TF-IDF is significantly better than Bag of Words** for sentiment analysis -- it downweights common words that appear in every tweet regardless of sentiment.

4. **Word clouds reveal clear sentiment vocab** -- certain words like 'hate', 'miss', 'sad' are almost exclusive to negative tweets, while 'love', 'thank', 'great' strongly indicate positive sentiment.

5. **Error analysis shows the limits of bag-of-words models** -- sarcasm, negation, and slang remain challenging.

### Ideas for Improvement

| Approach | Expected Gain | Complexity |
|:---------|:-------------:|:----------:|
| Tune TF-IDF parameters (max_features, ngram_range) | Small | Low |
| Use word embeddings (Word2Vec, GloVe) instead of TF-IDF | Medium | Medium |
| Try XGBoost or Random Forest on TF-IDF features | Small-Medium | Low |
| Add sentiment lexicon features (VADER, TextBlob) | Medium | Low |
| Use LSTM/Transformer models (BERT, DistilBERT) | Large | High |
| Handle negation explicitly (not good becomes not_good as bigram) | Medium | Low |
| Keep emoji/special characters as features | Small | Low |


### Detailed Classification Report

For completeness, here is the full scikit-learn classification report showing per-class metrics.


In [ ]:
print('=' * 65)
print('LOGISTIC REGRESSION -- Classification Report')
print('=' * 65)
print(classification_report(y_test, y_pred_lr, target_names=['Negative', 'Positive']))
print()
print('=' * 65)
print('NAIVE BAYES -- Classification Report')
print('=' * 65)
print(classification_report(y_test, y_pred_nb, target_names=['Negative', 'Positive']))
